In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [ ]:
class Head(nn.Module):

    def __init__(self, x_emb, head_emb, masking_enabled = False):
        super.__init__()
        self.key = nn.Linear(x_emb, head_emb)
        self.query = nn.Linear(x_emb, head_emb)
        self.value = nn.Linear(x_emb, head_emb)
        self.masking_enabled = masking_enabled

    def forward(self, x, key = None, query = None):
        k = self.key(x)                         # (batch_size, seq_len, head_emb)
        q = self.query(x)                       # (batch_size, seq_len, head_emb)
        v = self.value(x)                       # (batch_size, seq_len, head_emb)

        logits = q @ k.transpose(-2, -1)        # (batch_size, seq_len, head_emb)
        logits = logits / (k.shape[1] ** 0.5)

        # TODO: masking

        logits = F.softmax(logits)
        logits = logits @ v
        return logits

In [ ]:
class CrossHead(nn.Module):

    def __init__(self, x_emb, head_emb, masking_enabled = False):
        super.__init__()
        self.value = nn.Linear(x_emb, head_emb)
        self.masking_enabled = masking_enabled

    def forward(self, x, key, query):
        k = key(x)                         # (batch_size, seq_len, head_emb)
        q = query(x)                       # (batch_size, seq_len, head_emb)
        v = self.value(x)                       # (batch_size, seq_len, head_emb)

        logits = q @ k.transpose(-2, -1)        # (batch_size, seq_len, head_emb)
        logits = logits / (k.shape[1] ** 0.5)

        # TODO: masking

        logits = F.softmax(logits)
        logits = logits @ v
        return logits

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, x_emb, head_emb, heads_num, cross_attention = False, masking_enabled = False):
        super.__init__()
        self.proj = nn.Linear(x_emb, x_emb)
        HeadClass = CrossHead if cross_attention else Head
        self.heads = [HeadClass(x_emb//heads_num, head_emb, masking_enabled) for _ in range(heads_num)]

    def forward(self, x):
        logits = torch.concat(head(x[:, :, i * len(self.heads): (i+1) * len(self.heads)]) for i, head in enumerate(self.heads))
        logits = self.proj(logits)
        return logits